# ClustMRF on ClueWeb09 / TREC Web Track

**Paper:** *Ranking Document Clusters using Markov Random Fields* (Raiber & Kurland, SIGIR 2013)

**Collections (Table 2 of paper):**

| Variant | Corpus | Spam filter? |
|---------|--------|-------------|
| ClueA   | ClueWeb09 Category A (English, ~500M docs) | No |
| ClueAF  | ClueWeb09 Category A | Yes — Waterloo spam filter |
| ClueB   | ClueWeb09 Category B (~50M docs, subset of A) | No |
| ClueBF  | ClueWeb09 Category B | Yes — Waterloo spam filter |

**Topics / Qrels:** TREC Web Track 2009–2012 (adhoc task), pooled.

**Initial retrieval:** SDM (Metzler & Croft 2005), λ_T=0.85/λ_O=0.10/λ_U=0.05, DirichletLM μ=2500.

**Web lC features added (beyond non-web ClustMRF):**

| Feature | Description |
|---------|-------------|
| P_spam  | Waterloo spam score ∈ [0,1] (1 = not spam) |
| P_pr    | PageRank score (raw; log-transformed) |
| P_urlslash | URL path depth (# slashes) |
| P_urllen   | URL character length |

For each web measure, three aggregators are computed: geo / min / max.
This notebook runs this session's configured collection (A or B) and produces
both the vanilla and spam-filtered (F) variants in one pass.

**Prerequisites:**
1. ClueWeb09 data available to ir-datasets (set `CLUEWEB09_HOME` env var or see
   [ir-datasets ClueWeb09 setup](https://ir-datasets.com/clueweb09.html)).
2. Waterloo spam scores downloaded (see Cell 7).
3. PageRank file (optional, see Cell 11).
4. Terrier index with `blocks=True` is built automatically on first run.
5. Java 21 installed (`brew install --cask temurin@21`).

## 1  Configuration

In [ ]:
# ── Choose collection ─────────────────────────────────────────────────────────
# 'A' = ClueWeb09 Category A (English, ~500M docs)
# 'B' = ClueWeb09 Category B (~50M docs, subset of A)
COLLECTION = 'B'   # change to 'A' for the full-category experiments

# TREC Web Track years to pool
YEARS = [2009, 2010, 2011, 2012]

# Waterloo spam filter: path to the uncompressed score file.
# Score format (one doc per line): <score:int> <docid:str>  (score first!)
# Higher score = less likely to be spam (range 0-100).
# Download: http://plg.uwaterloo.ca/~gvcormac/clueweb09spam/
SPAM_SCORE_FILE = 'data/spam/waterloo-spam-clueweb09.txt'  # set to None to skip

# Spam filter threshold: keep docs with score >= threshold (paper uses ~50).
SPAM_THRESHOLD_RAW = 50   # on 0-100 scale

# PageRank file (optional): path to a file with lines "<docid>\t<pr_score>".
# Set to None to disable PageRank features.
PAGERANK_FILE = None   # e.g., 'data/pagerank/clueweb09-pagerank.txt'

# Background docs per shard for the judgment-pool index (set 0 to skip background)
N_BACKGROUND = 0   # ClueWeb09 qrel pool is large; background sampling is optional

print(f'Collection : ClueWeb09-{COLLECTION}')
print(f'Years      : {YEARS}')
print(f'Spam file  : {SPAM_SCORE_FILE}')
print(f'PR file    : {PAGERANK_FILE}')

## 2  Environment Setup

In [ ]:
import os, sys, warnings, pathlib, logging, shutil
warnings.filterwarnings('ignore')

os.environ.setdefault(
    'JAVA_HOME',
    '/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home'
)

ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import ir_datasets
import ir_measures
from ir_measures import MAP, nDCG, P
import pyterrier as pt

if not pt.java.started():
    pt.java.set_memory_limit(4096)
    pt.java.init()

logging.basicConfig(level=logging.WARNING)

_COLL_TAG = {'A': 'en', 'B': 'catb'}[COLLECTION]
INDEX_DIR  = ROOT / 'data' / 'indexes' / f'trec-web-pool-clue{COLLECTION}'
RUNS_DIR   = ROOT / 'data' / 'runs'    / f'trec-web-clue{COLLECTION}'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PyTerrier  : {pt.__version__}')
print(f'ir_datasets: {ir_datasets.__version__}')
print(f'Index dir  : {INDEX_DIR.relative_to(ROOT)}')

## 3  Topics and Qrels (Pool 2009–2012)

In [ ]:
topics_rows = []
qrels_rows  = []
qrel_docids = set()

for year in YEARS:
    ds_id = f'clueweb09/{_COLL_TAG}/trec-web-{year}'
    try:
        ds = ir_datasets.load(ds_id)
    except Exception as e:
        print(f'  WARNING: could not load {ds_id}: {e}')
        continue

    # Topics — use 'title' query ("query" field in ir-datasets)
    for q in ds.queries_iter():
        topics_rows.append({'qid': f'{year}-{q.query_id}', 'query': q.query})

    # Qrels
    for r in ds.qrels_iter():
        qid = f'{year}-{r.query_id}'
        qrels_rows.append({'query_id': qid, 'doc_id': r.doc_id, 'relevance': r.relevance})
        qrel_docids.add(r.doc_id)

    print(f'  {year}: {ds.queries_count()} topics, {ds.qrels_count()} qrel rows')

topics_df = pd.DataFrame(topics_rows).drop_duplicates('qid').reset_index(drop=True)
qrels_df  = pd.DataFrame(qrels_rows)

print(f'\nTotal topics    : {len(topics_df)}')
print(f'Total qrel rows : {len(qrels_df):,}')
print(f'Unique doc IDs  : {len(qrel_docids):,}')
print()
print('Relevance distribution:')
print(qrels_df['relevance'].value_counts().sort_index().to_string())
topics_df.head()

## 4  Load Waterloo Spam Scores

The Waterloo spam scores assign each ClueWeb09 document a score in [0, 100]:
- **100** = very unlikely to be spam
- **0** = very likely to be spam

Two uses in ClustMRF:
1. **Spam filter (F variants)**: docs with score < `SPAM_THRESHOLD_RAW` are removed from the SDM initial list.
2. **P_spam feature**: normalized score ∈ [0, 1] used as a cluster-level lC feature.

File format (one doc per line): `<score:int> <docid:str>` (score **first**!).

Download the score file from the Waterloo Spam Research Group (a single ~2 GB file
for the full ClueWeb09 collection, or a smaller one for Category B).

In [ ]:
spam_scores: dict[str, float] = {}   # {docno: score ∈ [0,1]}
spam_scores_raw: dict[str, int] = {} # {docno: score ∈ [0,100]}

spam_path = pathlib.Path(SPAM_SCORE_FILE) if SPAM_SCORE_FILE else None

if spam_path and spam_path.exists():
    print(f'Loading spam scores from {spam_path} …')
    with open(spam_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                score_raw, docid = int(parts[0]), parts[1]
                spam_scores_raw[docid]  = score_raw
                spam_scores[docid]      = score_raw / 100.0

    # Summary for qrel docs only
    qrel_spam = [spam_scores_raw.get(d) for d in qrel_docids
                 if d in spam_scores_raw]
    n_covered = len(qrel_spam)
    print(f'Loaded       : {len(spam_scores):,} scores total')
    print(f'Qrel covered : {n_covered:,} / {len(qrel_docids):,}')
    if qrel_spam:
        arr = np.array(qrel_spam)
        print(f'Score range  : {arr.min()}–{arr.max()}  (mean {arr.mean():.1f})')
        n_spam = (arr < SPAM_THRESHOLD_RAW).sum()
        print(f'Would filter : {n_spam:,} docs (score < {SPAM_THRESHOLD_RAW})')
elif SPAM_SCORE_FILE:
    print(f'WARNING: spam file not found at {SPAM_SCORE_FILE}')
    print('  Spam features and spam filter will be disabled.')
else:
    print('No spam file configured — spam features and spam filter disabled.')

SPAM_AVAILABLE = bool(spam_scores)
print(f'\nSpam features available: {SPAM_AVAILABLE}')

## 5  Build Judgment-Pool Index

We fetch every qrel-referenced document by ID from the ir-datasets docs store and
build a Terrier index with `blocks=True` (required for SDM proximity scoring).

URL features (P_urlslash, P_urllen) are computed at index time and cached
alongside the index so they don't need a second pass.

**First run:** fetches up to ~60K documents and builds the index — expect a few minutes.
**Subsequent runs:** loads the cached index in seconds.

In [ ]:
import json
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from src.algorithms.clust_mrf import url_features

INDEX_DIR.mkdir(parents=True, exist_ok=True)
props_file     = INDEX_DIR / 'data.properties'
blocks_marker  = INDEX_DIR / '.blocks_enabled'
url_feats_file = INDEX_DIR / 'url_features.json'   # cached URL features


def _extract_text(html: str) -> str:
    """Strip HTML tags, return plain text (title + body)."""
    soup = BeautifulSoup(html or '', 'html.parser')
    for tag in soup(['script', 'style', 'noscript', 'nav', 'header', 'footer']):
        tag.decompose()
    return ' '.join(soup.get_text(' ', strip=True).split()[:2000])  # cap at 2K words


def corpus_iter(store, docids):
    """Yield {docno, text} dicts from an ir-datasets docs_store, batch-fetching."""
    batch_size = 1000
    docids_list = list(docids)
    url_feat_cache = {}
    for start in tqdm(range(0, len(docids_list), batch_size),
                      desc='Fetching docs', unit='batch'):
        batch = docids_list[start:start + batch_size]
        docs  = store.get_many(batch)
        for docid in batch:
            doc = docs.get(docid)
            if doc is None:
                continue   # not in this collection (e.g., A-only doc when using B store)
            text = _extract_text(doc.body)
            url  = getattr(doc, 'url', '') or ''
            url_feat_cache[docid] = url_features(url)
            yield {'docno': docid, 'text': text}
    # Persist URL features alongside index
    with open(url_feats_file, 'w') as f:
        json.dump(url_feat_cache, f)
    print(f'URL features cached for {len(url_feat_cache):,} docs')


# Rebuild if existing index lacks blocks
if props_file.exists() and not blocks_marker.exists():
    print('Existing index has no blocks — rebuilding for SDM support…')
    shutil.rmtree(str(INDEX_DIR))
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

if not props_file.exists():
    # Use the first available dataset year's store for document access
    _ref_ds_id = f'clueweb09/{_COLL_TAG}/trec-web-{YEARS[0]}'
    print(f'Opening docs store via {_ref_ds_id} …')
    _ref_ds   = ir_datasets.load(_ref_ds_id)
    _store    = _ref_ds.docs_store()

    print(f'Building judgment-pool index for {len(qrel_docids):,} docs …')
    indexer = pt.IterDictIndexer(
        str(INDEX_DIR),
        overwrite  = True,
        meta       = {'docno': 40, 'text': 8192},
        text_attrs = ['text'],
        blocks     = True,
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
    )
    indexer.index(corpus_iter(_store, qrel_docids))
    blocks_marker.touch()
    print('Index built.')
else:
    print(f'Index exists at {INDEX_DIR} — loading.')

index = pt.IndexFactory.of(str(props_file))
stats = index.getCollectionStatistics()
print(f'Documents    : {stats.numberOfDocuments:,}')
print(f'Tokens       : {stats.numberOfTokens:,}')
print(f'Unique terms : {stats.numberOfUniqueTerms:,}')

## 6  Load URL Features

In [ ]:
import json

url_feat_dict: dict[str, dict] = {}
if url_feats_file.exists():
    with open(url_feats_file) as f:
        url_feat_dict = json.load(f)
    print(f'URL features loaded for {len(url_feat_dict):,} docs')
    # Sanity check
    example = next(iter(url_feat_dict.items()))
    print(f'Example: {example[0]} → {example[1]}')
else:
    print('URL feature file not found — URL features disabled.')
    print(f'  (Expected at {url_feats_file})')

## 7  Load PageRank (Optional)

PageRank is a query-independent lC feature: `P_pr(d)` = raw PageRank value.
ClustMRF applies `log(pr + ε)` internally, so raw values can be used directly.

File format: `<docid>\t<pr_score>` (one doc per line, tab-separated).

If `PAGERANK_FILE` is `None`, PageRank features are skipped gracefully.

In [ ]:
pagerank: dict[str, float] = {}

pr_path = pathlib.Path(PAGERANK_FILE) if PAGERANK_FILE else None

if pr_path and pr_path.exists():
    print(f'Loading PageRank from {pr_path} …')
    with open(pr_path) as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                docid, pr_val = parts[0], float(parts[1])
                pagerank[docid] = pr_val
    print(f'Loaded {len(pagerank):,} PageRank scores')
elif PAGERANK_FILE:
    print(f'WARNING: PageRank file not found at {PAGERANK_FILE}')
else:
    print('No PageRank file configured — PR features disabled.')

PR_AVAILABLE = bool(pagerank)
print(f'PageRank features available: {PR_AVAILABLE}')

## 8  Assemble doc_features Dict

Merge spam scores, PageRank, and URL features into a single `{docno: {feat: val}}` dict
that `ClustMRF` will look up during cluster scoring.

In [ ]:
# Build the combined doc_features dict
# Only include docs that actually appear in the index (from qrel pool)
doc_features: dict[str, dict] = {}

all_docids = set(url_feat_dict.keys()) | set(spam_scores.keys()) | set(pagerank.keys())

for docid in all_docids:
    entry: dict = {}
    if docid in spam_scores:
        entry['spam'] = spam_scores[docid]       # [0,1]
    if docid in pagerank:
        entry['pr']   = pagerank[docid]           # raw PageRank
    if docid in url_feat_dict:
        entry.update(url_feat_dict[docid])        # urlslash, urllen
    if entry:
        doc_features[docid] = entry

print(f'doc_features entries : {len(doc_features):,}')
active_feats = set()
for v in list(doc_features.values())[:100]:
    active_feats.update(v.keys())
print(f'Feature keys present : {sorted(active_feats)}')
if doc_features:
    sample = next(iter(doc_features.items()))
    print(f'Example : {sample[0]} → {sample[1]}')

## 9  SDM Initial Retrieval

In [ ]:
MEASURES = [MAP @ 100, P @ 5, nDCG @ 5, nDCG @ 10, nDCG @ 20]

sdm_rewrite  = pt.rewrite.SDM()   # Terrier defaults: λ_T=0.85, λ_O=0.10, λ_U=0.05
dirichlet_br = pt.BatchRetrieve(
    index,
    wmodel      = 'DirichletLM',
    num_results = 100,
    metadata    = ['docno', 'text'],
    controls    = {'c': 2500},
    verbose     = True,
)
sdm_pipe = sdm_rewrite >> dirichlet_br

print(f'Running SDM + DirichletLM on {len(topics_df)} topics…')
sdm_run = sdm_pipe.transform(topics_df)
print(f'Retrieved {len(sdm_run):,} rows  ({sdm_run["qid"].nunique()} queries)')
sdm_run.head()

## 10  Baseline Evaluation (SDM)

In [ ]:
sdm_eval = sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
sdm_agg  = ir_measures.calc_aggregate(MEASURES, qrels_df, sdm_eval)

print(f'SDM Baseline — ClueWeb09-{COLLECTION} / TREC Web Track {YEARS[0]}–{YEARS[-1]}')
print('=' * 52)
for m in MEASURES:
    print(f'  {str(m):<15} {float(sdm_agg[m]):.4f}')

## 11  Spam-Filtered Initial List (F Variants)

For ClueXF (X=A or B), documents predicted as spam by the Waterloo filter are
removed from the SDM initial list **before** ClustMRF re-ranking.
Documents without a spam score are kept (conservative).

In [ ]:
SPAM_THRESH_NORM = SPAM_THRESHOLD_RAW / 100.0

if SPAM_AVAILABLE:
    # Keep doc if spam_score >= threshold OR no score available (default: keep)
    sdm_run_f = sdm_run[
        sdm_run['docno'].map(spam_scores).fillna(SPAM_THRESH_NORM) >= SPAM_THRESH_NORM
    ].copy()
    sdm_run_f = sdm_run_f.reset_index(drop=True)

    n_removed = len(sdm_run) - len(sdm_run_f)
    print(f'Spam filter (threshold={SPAM_THRESHOLD_RAW}): removed {n_removed:,} rows '
          f'({n_removed/len(sdm_run)*100:.1f}%)')

    # Re-rank within query groups
    sdm_run_f['rank'] = (
        sdm_run_f.groupby('qid')['score']
        .rank(ascending=False, method='first')
        .astype(int)
    )

    sdm_eval_f = sdm_run_f.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    sdm_agg_f  = ir_measures.calc_aggregate(MEASURES, qrels_df, sdm_eval_f)

    print(f'\nSDM+SpamFilter baseline:')
    for m in MEASURES:
        print(f'  {str(m):<15} {float(sdm_agg_f[m]):.4f}')
else:
    sdm_run_f  = sdm_run.copy()   # fallback: unfiltered
    sdm_agg_f  = sdm_agg
    sdm_eval_f = sdm_eval
    print('Spam scores not available — spam filter variants will use unfiltered SDM list.')

## 12  ClustMRF — Weight Configurations

The paper learns weights with SVM^rank.  Since we don't have a training set here,
we use two pre-set configurations:

**`WEIGHTS_BASE`** — 13 non-web features only (same as robust04_clustmrf.ipynb).

**`WEIGHTS_WEB`** — all available web features added.  The 13 non-web weights are
scaled down proportionally so all active weights still sum to 1.0.  Web feature
weights are equal among the active features.

Tune these weights (or train with SVM^rank) for best results on held-out queries.

In [ ]:
# Non-web weights (proportional to Table 3 importance, non-web setting)
WEIGHTS_BASE = dict(
    w_geo_qsim=0.132, w_stdv_qsim=0.143, w_max_qsim=0.121, w_min_qsim=0.088,
    w_min_dsim=0.110, w_max_dsim=0.066, w_geo_dsim=0.055,
    w_min_icompress=0.099, w_geo_icompress=0.077, w_max_icompress=0.044,
    w_geo_entropy=0.033, w_min_entropy=0.022, w_max_entropy=0.011,
)

# Count active web features
_web_feat_names = [
    *(['spam'] * 3 if SPAM_AVAILABLE  else []),
    *(['pr']   * 3 if PR_AVAILABLE    else []),
    *(['url']  * 6 if url_feat_dict   else []),  # urlslash × 3 + urllen × 3
]
n_web = len(_web_feat_names)
n_nonweb = 13

if n_web > 0:
    # Scale non-web weights to occupy 65% of total weight budget
    scale = 0.65
    nw_total = sum(WEIGHTS_BASE.values())
    scaled_base = {k: v * scale / nw_total for k, v in WEIGHTS_BASE.items()}
    w_per_web  = (1.0 - scale) / n_web

    WEIGHTS_WEB = dict(scaled_base)
    if SPAM_AVAILABLE:
        WEIGHTS_WEB.update(w_geo_spam=w_per_web, w_min_spam=w_per_web, w_max_spam=w_per_web)
    if PR_AVAILABLE:
        WEIGHTS_WEB.update(w_geo_pr=w_per_web, w_min_pr=w_per_web, w_max_pr=w_per_web)
    if url_feat_dict:
        WEIGHTS_WEB.update(
            w_geo_urlslash=w_per_web, w_min_urlslash=w_per_web, w_max_urlslash=w_per_web,
            w_geo_urllen=w_per_web,   w_min_urllen=w_per_web,   w_max_urllen=w_per_web,
        )
    total_web = sum(WEIGHTS_WEB.values())
    print(f'Web weight config: {n_nonweb} non-web + {n_web} web features')
    print(f'  Non-web budget : {scale:.0%}  Web budget : {1-scale:.0%}')
    print(f'  Weight sum     : {total_web:.3f}')
else:
    WEIGHTS_WEB = WEIGHTS_BASE.copy()
    print('No web features available — WEIGHTS_WEB = WEIGHTS_BASE')

## 13  Run All Four Variants

| Variant | Initial list | Weights |
|---------|-------------|--------|
| Clue{X} (base)     | SDM unfiltered | non-web only |
| Clue{X} (web)      | SDM unfiltered | non-web + web features |
| Clue{X}F (base)    | SDM spam-filtered | non-web only |
| Clue{X}F (web)     | SDM spam-filtered | non-web + web features |

In [ ]:
import time
from src.algorithms.clust_mrf import ClustMRF

X = COLLECTION   # 'A' or 'B'

variants = [
    # (label,             init_run,    weight_config, doc_features_arg)
    (f'Clue{X} (base)',  sdm_run,     WEIGHTS_BASE,  None),
    (f'Clue{X} (web)',   sdm_run,     WEIGHTS_WEB,   doc_features if doc_features else None),
    (f'Clue{X}F (base)', sdm_run_f,   WEIGHTS_BASE,  None),
    (f'Clue{X}F (web)',  sdm_run_f,   WEIGHTS_WEB,   doc_features if doc_features else None),
]

results: dict[str, pd.DataFrame] = {}  # label → run DataFrame
evals:   dict[str, dict]         = {}  # label → aggregated metrics

for label, init_run, weights, df_arg in variants:
    print(f'\n{label} …')
    cmrf = ClustMRF(
        index        = index,
        k            = 5,
        n_docs       = 100,
        doc_features = df_arg,
        **weights,
    )
    t0  = time.time()
    run = cmrf.transform(init_run)
    elapsed = time.time() - t0
    print(f'  {elapsed:.1f}s  ({elapsed/init_run["qid"].nunique():.2f}s/query)')

    ev  = run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    agg = ir_measures.calc_aggregate(MEASURES, qrels_df, ev)
    for m in MEASURES:
        print(f'  {str(m):<15} {float(agg[m]):.4f}')

    results[label] = run
    evals[label]   = agg

## 14  Results Table

In [ ]:
rows = []
# Baselines
for name, agg in [(f'SDM (init)', sdm_agg),
                   (f'SDM+Filter', sdm_agg_f)]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    rows.append(row)

# ClustMRF variants
for label, agg in evals.items():
    row = {'System': f'ClustMRF [{label}]'}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    rows.append(row)

results_table = pd.DataFrame(rows).set_index('System')

print(f'\n=== ClueWeb09-{COLLECTION} / TREC Web Track {YEARS[0]}–{YEARS[-1]} ===')
print(results_table.to_string())

## 15  Per-Query MAP Analysis (ClueXF web vs SDM+Filter)

In [ ]:
best_variant = f'Clue{X}F (web)'
best_run = results[best_variant]

sdm_f_perq = ir_measures.iter_calc([MAP], qrels_df, sdm_eval_f)
best_perq  = ir_measures.iter_calc(
    [MAP], qrels_df, best_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
)

sdm_f_map_q = {r.query_id: r.value for r in sdm_f_perq  if r.measure == MAP}
best_map_q  = {r.query_id: r.value for r in best_perq   if r.measure == MAP}

perq_rows = [
    {'qid': qid,
     'SDM+Filter': round(sdm_f_map_q.get(qid, 0.0), 4),
     best_variant: round(best_map_q.get(qid, 0.0), 4)}
    for qid in sorted(sdm_f_map_q)
]
perq_df = pd.DataFrame(perq_rows)
perq_df['Δ MAP'] = (perq_df[best_variant] - perq_df['SDM+Filter']).round(4)

wins   = (perq_df['Δ MAP'] > 0).sum()
losses = (perq_df['Δ MAP'] < 0).sum()
ties   = (perq_df['Δ MAP'] == 0).sum()

print(f'{best_variant} vs SDM+Filter:')
print(f'  Wins   : {wins}   Losses : {losses}   Ties : {ties}')
print()
print('Top-5 gains:')
print(perq_df.nlargest(5, 'Δ MAP')[['qid','SDM+Filter', best_variant,'Δ MAP']].to_string(index=False))
print()
print('Top-5 losses:')
print(perq_df.nsmallest(5, 'Δ MAP')[['qid','SDM+Filter', best_variant,'Δ MAP']].to_string(index=False))

## 16  Ablation: Cluster Size k

In [ ]:
k_rows = []

for k_val in [3, 5, 10, 20]:
    cmrf_k = ClustMRF(
        index        = index,
        k            = k_val,
        n_docs       = 100,
        doc_features = doc_features if doc_features else None,
        **WEIGHTS_WEB,
    )
    run_k  = cmrf_k.transform(sdm_run_f)
    ev_k   = run_k.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
    agg_k  = ir_measures.calc_aggregate(MEASURES, qrels_df, ev_k)
    row    = {'System': f'ClustMRF k={k_val} (web, filtered)'}
    for m in MEASURES:
        row[str(m)] = round(float(agg_k[m]), 4)
    k_rows.append(row)
    print(f'  k={k_val}: done')

k_df = pd.DataFrame(k_rows).set_index('System')
print('\n=== Ablation: cluster size k ===')
print(k_df.to_string())

## 17  Save Runs (TREC Format)

In [ ]:
def save_trec_run(run_df: pd.DataFrame, path: pathlib.Path, tag: str) -> None:
    with open(path, 'w') as f:
        for qid, group in run_df.sort_values(['qid', 'rank']).groupby('qid'):
            for rank, row in enumerate(group.itertuples(), start=1):
                f.write(f'{qid} Q0 {row.docno} {rank} {row.score:.6f} {tag}\n')
    print(f'Saved: {path}')

save_trec_run(sdm_run,   RUNS_DIR / 'sdm.txt',        f'SDM_Clue{X}')
save_trec_run(sdm_run_f, RUNS_DIR / 'sdm_filter.txt', f'SDM_Clue{X}F')

for label, run in results.items():
    fname = label.lower().replace(' ', '_').replace('(', '').replace(')', '') + '.txt'
    save_trec_run(run, RUNS_DIR / fname, f'ClustMRF_{label.replace(" ","_")}')

---

## Summary

**Expected results (from Table 2 of Raiber & Kurland 2013):**

| System | MAP (approx.) |
|--------|---------------|
| ClueA  | ~0.090 |
| ClueAF | ~0.098 |
| ClueB  | ~0.113 |
| ClueBF | ~0.120 |

These are approximate; the paper uses SVM^rank-trained weights on a subset of queries.
Our fixed weights (proportional to Table 3 non-web importance + equal web weights)
are a reasonable starting point but will not replicate results exactly.

**Key observations to look for:**
- Spam filter (F variants) consistently helps: spam documents hurt cluster coherence
- ClueB > ClueA: smaller, more curated corpus → cleaner initial list
- Web features (spam, PR, URL) should help on navigational queries
- URL depth (urlslash) correlates with document authority

**Next steps to improve faithfulness:**
1. Train SVM^rank weights on a held-out query set (using `svmrank` or `lightgbm`)
2. Use Terrier's `DirichletLM` with the full ClueWeb09 collection (not just pool) for the initial list
3. Add PageRank features once a precomputed file is available
4. Tune the Dirichlet μ parameter (2500 is standard but may not be optimal for ClueWeb09)